# 03 – Baseline Models: ARIMA, Prophet, XGBoost

This notebook trains and evaluates three baseline forecasting models:
- **ARIMA** – classical statistical time-series model
- **Prophet** – Facebook/Meta time-series model with trend/seasonality decomposition
- **XGBoost** – gradient-boosted trees with engineered features

Predictions are saved for use in notebook 05 (ensemble).

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.data_loader import load_processed_data
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter
from src.config import RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plotter = Plotter()

In [2]:
# ── Load processed data ─────────────────────────────────────────────────────
train = load_processed_data('train')
val   = load_processed_data('val')
test  = load_processed_data('test')

print(f'Train: {len(train)} | Val: {len(val)} | Test: {len(test)}')

target_col = 'Close'
y_test = test[target_col]

Train: 1006 | Val: 252 | Test: 249


## A. ARIMA

In [ ]:
# ── ARIMA Model ──────────────────────────────────────────────────────────
from src.models.arima_model import ARIMAModel
from src.utils.metrics import calculate_metrics
from src.visualization.plotter import Plotter

plotter = Plotter()
arima_order = (2, 1, 0)
window = 252
arima = ARIMAModel(order=arima_order)



print(f"--- Training ARIMA (Walk-forward | pdq={arima_order} | window={window} | Metrics=Validation) ---")

arima_val_preds, arima_test_preds = arima.train_and_refit(
train, val, test, target_col="Close", window=window
)

print("\n [Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val['Close'], arima_val_preds))

print("\n [Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test['Close'], arima_test_preds))

# 参数标签：用于 title 和 filename
p, d, q = arima_order
pdq_tag = f"p{p}_d{d}_q{q}"
window_tag = f"w{window}"

# 2024 Validation 预测对比图
plotter.plot_predictions_comparison(
    y_true=val["Close"],
    predictions={"ARIMA Forecast": arima_val_preds},
    dates=val.index,
    title=f"ARIMA Validation Predictions (2024 | pdq={arima_order} | window={window})",
    filename=f"arima_val_2024_{pdq_tag}_{window_tag}.png"
)

# 2025 Test 预测对比图
plotter.plot_predictions_comparison(
    y_true=test["Close"],
    predictions={"ARIMA Forecast": arima_test_preds},
    dates=test.index,
    title=f"ARIMA Test Predictions (2025 | pdq={arima_order} | window={window})",
    filename=f"arima_test_2025_{pdq_tag}_{window_tag}.png"
)

## B. Prophet

In [ ]:
# ── 5. Prophet Model ────────────────────────────────────────────────────────
from src.models.prophet_model import ProphetModel

prophet = ProphetModel()
print("--- Training Prophet (Two-Phase) ---")
prophet_val_preds, prophet_test_preds = prophet.train_and_refit(train, val, test)

print("\n [Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val['Close'], prophet_val_preds))

print("\n [Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test['Close'], prophet_test_preds))

plotter.plot_predictions_comparison(
y_true=val["Close"],
predictions={"Prophet Forecast": prophet_val_preds},
dates=val.index,
title="Prophet 2024 Validation Predictions",
filename="prophet_val_forecast_2024.png"
)

plotter.plot_predictions_comparison(
    y_true=test['Close'],
    predictions={'Prophet Forecast': prophet_test_preds},
    dates=test.index,
    title="Prophet 2025 Test Predictions",
    filename="prophet_test_forecast_2025.png"
)

## C. XGBoost

In [4]:
# ── 6. XGBoost Model (Selective Shift-1 for OHLCV + Technical Indicators) ──
from pathlib import Path
import sys



cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(project_root))
print("cwd:", cwd)
print("project_root:", project_root)
from src.models.xgboost_model import XGBoostModel
import pandas as pd
import matplotlib.pyplot as plt

target_col = "Close"

# 候选特征：保留 Close 作为特征来源（会被 shift(1)），排除 Adj Close
base_features = [col for col in train.columns if col != "Adj Close"]

# 需要做 shift(1) 的特征集合
ohlcv_cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in base_features]
ma_cols = [c for c in base_features if c.startswith("MA")]
rsi_cols = [c for c in base_features if c == "RSI"]
macd_cols = [c for c in base_features if c in ["MACD", "MACD_signal", "MACD_hist"]]
bb_cols = [c for c in base_features if c.startswith("BB_")]

cols_to_shift = ohlcv_cols + ma_cols + rsi_cols + macd_cols + bb_cols
cols_to_shift = list(dict.fromkeys(cols_to_shift))  # 去重并保持顺序

print(f"Applying shift(1) to {len(cols_to_shift)} columns: OHLCV + MA/RSI/MACD/Bollinger")

def build_xgb_dataset(df: pd.DataFrame, all_features: list[str], shift_cols: list[str], target: str):
    X = df[all_features].copy()

    # 1) 仅指定列 shift(1) 并重命名
    X_shift = X[shift_cols].shift(1)
    X_shift.columns = [f"{c}_lag1" for c in shift_cols]

    # 2) 其余列保持原值（例如 lag_1~lag_5、宏观 Lag35、sentiment 等）
    keep_cols = [c for c in all_features if c not in shift_cols and c != target]
    X_keep = X[keep_cols].copy()

    # 3) 合并成最终特征
    X_final = pd.concat([X_shift, X_keep], axis=1)

    # shift 后首行 NaN 兜底
    X_final.bfill(inplace=True)

    # 拼回目标列（目标保持原时点，不shift）
    out_df = pd.concat([X_final, df[target]], axis=1)
    feature_cols = list(X_final.columns)
    return out_df, feature_cols

train_xgb_df, xgb_feature_cols = build_xgb_dataset(train, base_features, cols_to_shift, target_col)
val_xgb_df, _ = build_xgb_dataset(val, base_features, cols_to_shift, target_col)
test_xgb_df, _ = build_xgb_dataset(test, base_features, cols_to_shift, target_col)

print(f"XGBoost uses {len(xgb_feature_cols)} features.")
print("Shifted columns are suffixed with _lag1.")

# 4. 两阶段训练
xgb = XGBoostModel()
print("--- Training XGBoost (Two-Phase Refitting) ---")
xgb_val_preds, xgb_test_preds = xgb.train_and_refit(
    train_xgb_df, val_xgb_df, test_xgb_df,
    features=xgb_feature_cols,
    target_col=target_col
)
print(xgb.params)
print({k: xgb._model.get_xgb_params().get(k) for k in ["max_depth","learning_rate","min_child_weight","gamma","reg_alpha","reg_lambda"]})

print("\n[Phase 1] 2024 Validation Metrics:")
print(calculate_metrics(val["Close"], xgb_val_preds))

print("\n[Phase 2] 2025 Test Metrics:")
print(calculate_metrics(test["Close"], xgb_test_preds))

plotter.plot_predictions_comparison(
    y_true=val["Close"],
    predictions={"XGBoost Forecast": xgb_val_preds},
    dates=val.index,
    title="XGBoost 2024 Validation Predictions",
    filename="xgboost_val_forecast_2024.png",
)

plotter.plot_predictions_comparison(
    y_true=test["Close"],
    predictions={"XGBoost Forecast": xgb_test_preds},
    dates=test.index,
    title="XGBoost 2025 Test Predictions",
    filename="xgboost_test_forecast_2025.png",
)

# ---- Phase 1-only model for feature importance (train) ----
xgb_phase1 = XGBoostModel()
xgb_phase1.fit(
    train_xgb_df[xgb_feature_cols],
    train_xgb_df[target_col],
    X_val=val_xgb_df[xgb_feature_cols],
    y_val=val_xgb_df[target_col],
)

xgb_importance_train = pd.Series(
    xgb_phase1._model.feature_importances_,
    index=xgb_feature_cols
)

plotter.plot_feature_importance(
    xgb_importance_train,
    top_n=20,
    title="XGBoost Feature Importance (train)",
    filename="xgboost_feature_importance_train_2020-2023.png",
    annotate=True,
    decimals=4,
)

# ---- Phase 2 refitted importance ----
xgb_importance_refit = pd.Series(
    xgb._model.feature_importances_,
    index=xgb_feature_cols
)

plotter.plot_feature_importance(
    xgb_importance_refit,
    top_n=20,
    title="XGBoost Feature Importance (Refitted)",
    filename="xgboost_feature_importance_refit_2024-2025.png",
    annotate=True,
    decimals=4,
)

cwd: /workspaces/COMP5152ADA_Project_2/notebooks
project_root: /workspaces/COMP5152ADA_Project_2
Applying shift(1) to 16 columns: OHLCV + MA/RSI/MACD/Bollinger
XGBoost uses 31 features.
Shifted columns are suffixed with _lag1.
--- Training XGBoost (Two-Phase Refitting) ---
{'n_estimators': 2000, 'max_depth': 5, 'learning_rate': 0.05, 'subsample': 1, 'colsample_bytree': 1, 'min_child_weight': 1, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 1, 'random_state': 42, 'objective': 'reg:squarederror', 'early_stopping_rounds': 120}
{'max_depth': 5, 'learning_rate': 0.05, 'min_child_weight': 1, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 1}

[Phase 1] 2024 Validation Metrics:
{'mse': 45030.89763701845, 'rmse': 212.2048482882011, 'mae': 158.9957553966517, 'mape': 0.8338717844317705, 'directional_accuracy': 0.5697211155378487}

[Phase 2] 2025 Test Metrics:
{'mse': 94177.31865396386, 'rmse': 306.88323293064394, 'mae': 216.51810499211402, 'mape': 0.9926004930560584, 'directional_accuracy': 0.487903225806

## D. Comparison

In [ ]:
# ── 7. Aggregate and Save Final Test Metrics ─────────────────────────────
from src.utils.metrics import calculate_metrics
import pandas as pd
from src.config import RESULTS_DIR

# 重新计算各模型在 Phase 2 (2025 测试集) 上的最终表现
arima_metrics = calculate_metrics(test['Close'], arima_test_preds)
prophet_metrics = calculate_metrics(test['Close'], prophet_test_preds)
xgb_metrics = calculate_metrics(test['Close'], xgb_test_preds)

all_metrics = {
    'ARIMA': arima_metrics,
    'Prophet': prophet_metrics,
    'XGBoost': xgb_metrics,
}

# 转换为 DataFrame 方便展示和保存 (转置 T 是为了让模型名在行，指标在列)
metrics_df = pd.DataFrame(all_metrics).T

print("\n🏆 FINAL BASELINE METRICS (2025 Test Set - Refitted) 🏆")
print("=" * 70)
print(metrics_df.to_string())
print("=" * 70)

# 保存最终的指标表格
metrics_df.to_csv(RESULTS_DIR / 'baseline_metrics.csv')
print(f"\n✅ Metrics successfully saved to {RESULTS_DIR / 'baseline_metrics.csv'}")

# 可选：绘制所有 Baseline 模型的指标对比柱状图
plotter.plot_metrics_comparison(all_metrics, filename='baseline_metrics_comparison.png')

In [ ]:
# ── 8. Save Phase 1 & Phase 2 Predictions ────────────────────────────────
from src.config import RESULTS_DIR
import pandas as pd

# 1. 横向拼接各个模型的结果
val_preds_df = pd.concat([arima_val_preds, prophet_val_preds, xgb_val_preds], axis=1)
test_preds_df = pd.concat([arima_test_preds, prophet_test_preds, xgb_test_preds], axis=1)

# 2. 纵向拼接 2024(Val) 和 2025(Test) 的结果，合并为你原来熟悉的 preds_df
preds_df = pd.concat([val_preds_df, test_preds_df], axis=0)
# 重命名列，保持规范
preds_df.columns = ['ARIMA', 'Prophet', 'XGBoost']

# 3. 完美还原你原来的保存逻辑和打印语句
save_path = RESULTS_DIR / 'baseline_predictions.csv'
preds_df.to_csv(save_path)

print(f"Predictions successfully saved to {save_path}")
print(f"Total rows saved: {len(preds_df)} (Val: {len(val_preds_df)} + Test: {len(test_preds_df)})")

## Summary

See `reports/results/baseline_metrics.csv` for a full metrics table.

Continue to **04_model_lstm.ipynb** for the deep learning model.